In [20]:
import torch
from torch.utils.data import Dataset
import tiktoken
from typing import List, Tuple, Optional
import os

class TinyStoriesDataset(Dataset):
    """
    Датасет для обучения GPT-like модели на текстовых файлах TinyStories.

    Каждый текст в файле должен заканчиваться токеном <|endoftext|>.
    При достижении максимальной длины контекста (n токенов) кусок берется как семпл,
    а остаток переходит на следующий семпл.
    """

    def __init__(
        self,
        txt_file_path: str,
        max_length: int = 1024,
        tokenizer_name: str = "cl100k_base",  # или "p50k_base" для GPT-2/GPT-3
        overlap: int = 0,
        pad_token_id: Optional[int] = None
    ):
        self.max_length = max_length
        self.overlap = overlap

        # Инициализируем токенайзер
        self.tokenizer = tiktoken.get_encoding(tokenizer_name)

        # Получаем ID для <|endoftext|>
        # Для p50k_base и cl100k_base <|endoftext|> - это специальный токен
        try:
            self.eot_token_id = self.tokenizer.encode("<|endoftext|>", allowed_special="all")[0]
        except:
            # Если токен не найден, используем стандартный
            self.eot_token_id = self.tokenizer.eot_token

        # Устанавливаем pad_token_id (по умолчанию = eot_token_id)
        self.pad_token_id = pad_token_id if pad_token_id is not None else self.eot_token_id

        # Загружаем и токенизируем весь датасет
        print(f"Загрузка файла: {txt_file_path}")
        self.tokens = self._load_and_tokenize(txt_file_path)
        print(f"Всего токенов в датасете: {len(self.tokens):,}")

        # Создаем индексы для семплов
        self.sample_indices = self._create_sample_indices()
        print(f"Создано семплов: {len(self.sample_indices)}")

    def _load_and_tokenize(self, txt_file_path: str) -> List[int]:
        """Загружает txt файл и токенизирует его целиком."""
        with open(txt_file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        # Токенизируем весь текст сразу
        tokens = self.tokenizer.encode(text, allowed_special="all")
        return tokens

    def _create_sample_indices(self) -> List[Tuple[int, int]]:
        """
        Создает список индексов (start_idx, end_idx) для каждого семпла.
        Возвращает список кортежей.
        """
        indices = []
        i = 0
        total_tokens = len(self.tokens)

        while i < total_tokens:
            start_idx = i

            # Конец семпла (не может превышать total_tokens)
            end_idx = min(i + self.max_length, total_tokens)

            # Если достигли конца данных
            if start_idx >= total_tokens:
                break

            # Проверяем, есть ли eot внутри этого отрезка
            segment_tokens = self.tokens[start_idx:end_idx]

            # Ищем последний <|endoftext|> в сегменте
            last_eot_idx_in_segment = None
            for j, token in enumerate(segment_tokens):
                if token == self.eot_token_id:
                    last_eot_idx_in_segment = j

            # Если нашли eot внутри сегмента и это не конец сегмента
            if last_eot_idx_in_segment is not None:
                # Обрезаем семпл до позиции сразу после последнего eot
                actual_end_idx = start_idx + last_eot_idx_in_segment + 1
                indices.append((start_idx, actual_end_idx))

                # Следующий семпл начинается после eot (минус overlap)
                if self.overlap > 0:
                    i = actual_end_idx - self.overlap
                else:
                    i = actual_end_idx
            else:
                # Нет eot внутри max_length, берем полный кусок
                indices.append((start_idx, end_idx))

                # Двигаемся вперед с перекрытием
                if self.overlap > 0:
                    i = end_idx - self.overlap
                else:
                    i = end_idx

            # Защита от бесконечного цикла
            if i <= start_idx:
                i = start_idx + 1

        # Если индексы не созданы (пустой файл или ошибка), возвращаем пустой список
        if not indices:
            print("Предупреждение: не создано ни одного семпла!")
            return []

        return indices

    def __len__(self) -> int:
        return len(self.sample_indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Возвращает (input_ids, labels) где labels сдвинуты на 1 для next-token prediction.
        """
        if idx >= len(self.sample_indices):
            raise IndexError(f"Index {idx} out of range for dataset with {len(self.sample_indices)} samples")

        start_idx, end_idx = self.sample_indices[idx]

        # Получаем токены для этого семпла
        tokens = self.tokens[start_idx:end_idx].copy()  # копируем чтобы не изменять оригинал

        # Если семпл короче max_length, добавляем паддинги
        if len(tokens) < self.max_length:
            padding_length = self.max_length - len(tokens)
            tokens = tokens + [self.pad_token_id] * padding_length

        # Обрезаем если длиннее (на всякий случай)
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        # Конвертируем в тензор
        input_ids = torch.tensor(tokens, dtype=torch.long)

        # Создаем labels (сдвигаем на 1 вправо)
        # input_ids:  [t0, t1, t2, ..., t_{n-1}]
        # labels:     [t1, t2, t3, ..., pad]
        labels = torch.cat([input_ids[1:], torch.tensor([self.pad_token_id], dtype=torch.long)])

        # Для CrossEntropyLoss: заменяем pad_token_id на -100
        labels[labels == self.pad_token_id] = -100

        return input_ids, labels

    def get_tokenizer_info(self) -> dict:
        """Возвращает информацию о токенайзере."""
        return {
            "tokenizer_name": getattr(self.tokenizer, 'name', 'unknown'),
            "vocab_size": self.tokenizer.n_vocab,
            "eot_token_id": self.eot_token_id,
            "pad_token_id": self.pad_token_id,
            "max_length": self.max_length
        }

In [21]:

# Дополнительный коллатор для батчей
def collate_fn(batch):
    """
    Кастомный коллатор для объединения семплов в батч.
    Все input_ids и labels уже имеют одинаковую длину (max_length) благодаря паддингу.
    """
    input_ids = torch.stack([item[0] for item in batch])
    labels = torch.stack([item[1] for item in batch])
    return input_ids, labels

In [22]:
dataset = TinyStoriesDataset(
    txt_file_path='data/TinyStoriesV2-GPT4-valid.txt',
    max_length=256,
    tokenizer_name="p50k_base",
    overlap=0
)

print(f"Размер датасета: {len(dataset)}")
print(f"Информация о токенайзере: {dataset.get_tokenizer_info()}")

# Тестируем загрузку одного семпла
input_ids, labels = dataset[0]
print(f"Input shape: {input_ids.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Первые 10 input токенов: {input_ids[:10].tolist()}")
print(f"Первые 10 label токенов: {labels[:10].tolist()}")

Загрузка файла: data/TinyStoriesV2-GPT4-valid.txt
Всего токенов в датасете: 5,532,630
Создано семплов: 29651
Размер датасета: 29651
Информация о токенайзере: {'tokenizer_name': 'p50k_base', 'vocab_size': 50281, 'eot_token_id': 50256, 'pad_token_id': 50256, 'max_length': 256}
Input shape: torch.Size([256])
Labels shape: torch.Size([256])
Первые 10 input токенов: [84, 836, 470, 423, 284, 307, 12008, 286, 262, 7812]
Первые 10 label токенов: [836, 470, 423, 284, 307, 12008, 286, 262, 7812, 3290]


In [24]:

# Тестируем DataLoader
from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

for batch_input, batch_labels in dataloader:
    print(f"Batch input shape: {batch_input}")
    print(f"Batch labels shape: {batch_labels.shape}")
    break

Batch input shape: tensor([[  198,  7454,  2402,   257,   640,    11,   612,   373,   257,  1310,
          3290,  3706,  5436,    13,  5436,   373,   845,  6507,   290, 22444,
           780,   339,   550,   645,  2460,   284,   711,   351,    13,   679,
           561,  2513,  1088,   262,  3952,   477,  3436,    11,  2045,   329,
          2130,   284, 16225,   290,   423,  1257,   351,    13,   198,  3198,
          1110,    11,  5436,  2497,   257,  1310,  2576,  3706, 18966,    13,
         18966,   373,   635,  6507,   290, 22444,    11,  5586,   319,   257,
          7624,   477,   416,  5223,    13,  5436,  1807,   326,  3863,   611,
           339,  2921,   607,   257, 16225,    11,   673,   561,   307,   465,
          1545,    13,  1406,    11,   339,  6807,   510,   284,   607,   290,
         41130,   607,  1232,    13,   198, 10161,  2611,  2936,   262, 16225,
           290, 13541,    13,  1375, 41130,  5436,   736,    11,   290,   484,
          1111,  2936,  3772,    

In [25]:
encoder = tiktoken.get_encoding('p50k_base')